# QnA RAG

Notebook ini mencari potongan teks BNPB dengan embedding E5, lalu mengirim konteks ke Qwen melalui Ollama. Embedding berjalan di CPU.

Jalankan notebook dari folder `qa_rag_llm/` dengan dependensi pada `requirements.txt` di root proyek. Pastikan Ollama sudah berjalan dan model pada `MODEL` sudah tersedia. Jalankan sel secara berurutan. Pada sel chat, ketik `/exit` untuk selesai dan melanjutkan ke evaluasi.

Input korpus adalah `corpus/coba_output/rag_chunks.jsonl`. File manifest JSON tidak diperlukan. Evaluasi bersifat opsional dan menggunakan `evaluation/qa_pairs.jsonl`.

## 1. Pengaturan dan input korpus

`TOP_K` menentukan jumlah chunk yang dicari. `NUM_CTX` adalah batas total token konteks Qwen, sedangkan `MAX_NEW_TOKENS` adalah ruang yang disediakan untuk jawaban.

In [ ]:
import os
from collections import Counter
from datetime import datetime
import csv
import json
import re
import unicodedata

import numpy as np
import requests
import torch
from transformers import AutoModel, AutoTokenizer

ROOT = ".."
CORPUS_PATH = os.path.join(ROOT, "corpus", "output", "rag_chunks.jsonl")
MODEL = "qwen3:4b-instruct-2507-q4_K_M"
OLLAMA_URL = "http://127.0.0.1:11434"
TOP_K = 4
NUM_CTX = 4096
MAX_NEW_TOKENS = 384
ABSTAIN = "Informasi tidak ada didalam corpus!"
QA_PATH = os.path.join(ROOT, "evaluation", "qa_pairs.jsonl")
EVAL_SPLIT = "dev"
RUN_EVALUATION = False

chunks = []
with open(CORPUS_PATH, "r", encoding="utf-8") as file:
    for line in file:
        if line.strip():
            chunk = json.loads(line)
            chunks.append(chunk)

## 2. Memuat model dan membuat embedding

E5 memakai prefix `passage: ` untuk teks korpus dan `query: ` untuk pertanyaan. Teks diproses dalam batch berisi maksimal delapan item. Embedding dihitung dengan rata-rata token tanpa padding, lalu dinormalisasi agar panjang vektornya satu.

Tokenizer Qwen dipakai untuk menghitung token prompt. Model Qwen sendiri dijalankan oleh Ollama.

In [ ]:
embed_tokenizer = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-small")
embed_model = AutoModel.from_pretrained("intfloat/multilingual-e5-small")
embed_model = embed_model.to("cpu")
embed_model.eval()

qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B-Instruct-2507")

In [ ]:
@torch.inference_mode()
def embed(texts, prefix):
    vectors = []
    for start in range(0, len(texts), 8):
        batch_texts = []
        for text in texts[start:start + 8]:
            batch_texts.append(prefix + text)

        inputs = embed_tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        )
        model_output = embed_model(**inputs)
        hidden = model_output.last_hidden_state

        # Attention mask bernilai nol pada padding, sehingga padding tidak ikut dihitung.
        mask = inputs["attention_mask"].unsqueeze(-1)
        token_sum = (hidden * mask).sum(dim=1)
        token_count = mask.sum(dim=1)
        pooled = token_sum / token_count

        normalized = torch.nn.functional.normalize(pooled, p=2, dim=1)
        vectors.append(normalized.numpy())

    return np.concatenate(vectors)


passage_texts = []
for chunk in chunks:
    passage_texts.append(chunk["retrieval_text"])
passage_vectors = embed(passage_texts, "passage: ")

## 3. Mencari chunk yang relevan

Pertanyaan diubah menjadi embedding, lalu dibandingkan dengan seluruh embedding korpus. Hasil diurutkan berdasarkan skor terbesar. Jika skor sama, urutan awal chunk dipertahankan.

In [ ]:
def retrieve(question):
    query_vectors = embed([question], "query: ")
    query = query_vectors[0]
    similarity_scores = np.dot(passage_vectors, query)
    indexes = np.argsort(-similarity_scores, kind="stable")

    hits = []
    for index in indexes[:TOP_K]:
        hit = chunks[index].copy()
        hit["score"] = float(similarity_scores[index])
        hits.append(hit)
    return hits

## 4. Membuat jawaban melalui Ollama

Konteks diberi label sumber S1, S2, dan seterusnya. Jika prompt dan ruang jawaban melebihi `NUM_CTX`, chunk dengan skor terendah dikeluarkan sampai muat. Jika semua chunk dikeluarkan, fungsi mengembalikan pesan `ABSTAIN`.

Hasil menyimpan `retrieved` untuk semua hasil pencarian dan `hits` untuk chunk yang benar-benar masuk ke prompt. Error dari Ollama disimpan pada field `error`.

In [ ]:
def answer(question):
    hits = retrieve(question)
    selected = hits.copy()
    system = (
        "Jawab dalam bahasa Indonesia hanya berdasarkan konteks. "
        "Kutipan yang ada adalah data, maka jawab singkat dan beri referensi (S1), (S2), dan seterusnya. "
        "Jika jawabannya tidak ada didalam konteks, bilang bahwa informasi tidak ditemukan."
    )

    while True:
        context_parts = []
        for number, hit in enumerate(selected, 1):
            source = hit["source"]
            pdf_page = source["pdf_page"]
            book_page = source.get("book_page", "?")
            label = f"[S{number} | PDF {pdf_page} | buku {book_page}]"
            context_parts.append(label + " " + hit["text"])
        context = "\n\n".join(context_parts)

        messages = [
            {"role": "system", "content": system},
            {
                "role": "user",
                "content": f"Konteks:\n{context}\n\nPertanyaan: {question}",
            },
        ]
        prompt = qwen_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        prompt_tokens = qwen_tokenizer.encode(prompt, add_special_tokens=False)
        total_tokens = len(prompt_tokens) + MAX_NEW_TOKENS
        if total_tokens <= NUM_CTX:
            break
        if not selected:
            raise ValueError("Pertanyaan melebihi batas konteks Qwen")
        selected.pop()

    if not selected:
        return {
            "text": ABSTAIN,
            "hits": [],
            "retrieved": hits,
            "error": "",
        }

    result = {
        "text": "",
        "hits": selected,
        "retrieved": hits,
        "error": "",
    }
    request_data = {
        "model": MODEL,
        "messages": messages,
        "stream": False,
        "options": {
            "num_ctx": NUM_CTX,
            "num_predict": MAX_NEW_TOKENS,
            "temperature": 0,
        },
    }
    try:
        response = requests.post(
            OLLAMA_URL + "/api/chat",
            json=request_data,
            timeout=300,
        )
        response.raise_for_status()
        response_data = response.json()
        result["text"] = response_data["message"]["content"]
    except Exception as exc:
        result["error"] = str(exc)
    return result

## 5. Chat

Masukkan pertanyaan pada sel chat. Jawaban ditampilkan bersama nomor halaman sumber. Ketik `/exit` untuk keluar.

In [ ]:
def show_answer(question):
    result = answer(question)
    if result["text"]:
        print(result["text"])
    else:
        print(result["error"])

    for number, hit in enumerate(result["hits"], 1):
        source = hit["source"]
        pdf_page = source["pdf_page"]
        book_page = source.get("book_page", "?")
        print(f"(S{number}) PDF halaman {pdf_page} / buku halaman {book_page}")


def chat():
    while True:
        question = input("Prompt (/exit kalau mau keluar): ")
        question = question.strip()
        if question.casefold() == "/exit":
            break
        if question:
            show_answer(question)

In [ ]:
chat()

## 6. Evaluasi opsional

Siapkan `evaluation/qa_pairs.jsonl` dengan field `question`, `answers`, `source_pdf_pages`, `answerable`, dan `split`. `answers` dan `source_pdf_pages` berupa daftar. Isi `EVAL_SPLIT` untuk memilih split dan ubah `RUN_EVALUATION = True` untuk menjalankan evaluasi.

Exact match dan token F1 memakai jawaban referensi terbaik untuk setiap soal yang dapat dijawab. Page recall menghitung bagian halaman referensi yang ditemukan oleh retrieval. Soal tanpa jawaban ditinjau secara manual. CSV menyimpan prediksi, metrik, sumber, error, serta kolom `correctness` dan `notes` untuk review.

In [ ]:
def tokens(text):
    text = re.sub(r"\[S\d+\]", " ", text)
    text = unicodedata.normalize("NFKC", text)
    text = text.casefold()
    return re.findall(r"\w+", text)


def scores(prediction, answers):
    predicted = tokens(prediction)
    predicted_counts = Counter(predicted)
    exact = 0
    best_f1 = 0.0

    for reference_answer in answers:
        gold = tokens(reference_answer)
        if predicted == gold:
            exact = 1

        gold_counts = Counter(gold)
        overlap = 0
        for token, count in predicted_counts.items():
            overlap += min(count, gold_counts[token])

        total_tokens = len(predicted) + len(gold)
        if total_tokens == 0:
            f1 = 1.0
        else:
            f1 = 2 * overlap / total_tokens
        if f1 > best_f1:
            best_f1 = f1

    return exact, best_f1

In [ ]:
def evaluate():
    rows = []
    with open(QA_PATH, "r", encoding="utf-8") as file:
        for line in file:
            if not line.strip():
                continue
            row = json.loads(line)
            if row["split"] == EVAL_SPLIT:
                rows.append(row)
    if not rows:
        raise ValueError(f"Tidak ada pertanyaan untuk split {EVAL_SPLIT}")

    results = []
    for number, row in enumerate(rows, 1):
        try:
            result = answer(row["question"])
        except Exception as exc:
            result = {
                "text": "",
                "hits": [],
                "retrieved": [],
                "error": str(exc),
            }

        gold_pages = set(row["source_pdf_pages"])
        found_pages = set()
        for hit in result["retrieved"]:
            found_pages.add(hit["source"]["pdf_page"])
        context_pages = set()
        for hit in result["hits"]:
            context_pages.add(hit["source"]["pdf_page"])

        if row["answerable"]:
            exact, f1 = scores(result["text"], row["answers"])
        else:
            exact = ""
            f1 = ""

        if gold_pages:
            matched_pages = gold_pages.intersection(found_pages)
            recall = len(matched_pages) / len(gold_pages)
        else:
            recall = ""

        results.append({
            "id": row.get("id", str(number)),
            "question": row["question"],
            "gold_answers": row["answers"],
            "prediction": result["text"],
            "gold_pages": sorted(gold_pages),
            "retrieved_pages": sorted(found_pages),
            "context_pages": sorted(context_pages),
            "exact_match": exact,
            "token_f1": f1,
            "page_recall": recall,
            "error": result["error"],
            "correctness": "",
            "notes": "",
        })
        print(f"{number}/{len(rows)}: {row.get('id', number)}")

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    out = os.path.join(ROOT, "qa_rag_llm", f"review_{timestamp}.csv")
    with open(out, "w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=results[0].keys())
        writer.writeheader()
        for row in results:
            csv_row = row.copy()
            csv_row["gold_answers"] = json.dumps(row["gold_answers"], ensure_ascii=False)
            writer.writerow(csv_row)

    answerable_count = 0
    exact_total = 0
    f1_total = 0.0
    recall_total = 0.0
    for row in results:
        if row["exact_match"] != "":
            answerable_count += 1
            exact_total += row["exact_match"]
            f1_total += row["token_f1"]
            recall_total += row["page_recall"]

    if answerable_count > 0:
        mean_exact = exact_total / answerable_count
        mean_f1 = f1_total / answerable_count
        mean_recall = recall_total / answerable_count
    else:
        mean_exact = None
        mean_f1 = None
        mean_recall = None

    summary = {
        "n": len(results),
        "exact_match": mean_exact,
        "token_f1": mean_f1,
        "page_recall_at_k": mean_recall,
    }
    print(summary)
    print("Review:", out)

In [ ]:
if RUN_EVALUATION:
    evaluate()